# Noetrix: fine-tune Laya on vault passages (Kaggle, GPU T4 x2)

This notebook distils [Laya](https://github.com/NandhaKishorM/laya) (a ~421M-parameter encoder that answers a typed yes/no question about a text in one forward pass) from `BAAI/bge-reranker-v2-m3`, using passages exported from the Noetrix vault, and then scores the result on held-out passages against the teacher and against the `cross-encoder/ms-marco-MiniLM-L-6-v2` reranker the RAG ships.

**Read `docs/laya-finetune.md` first.** Its verdict is *spike-only*: a student distilled from a teacher is capped by that teacher, and zero-shot Laya scored below "no reranker" in three independent third-party benchmarks. This notebook finds out whether a fine-tune changes that on this vault; it does not assume it will.

**Notebook settings** (right-hand panel): Accelerator **GPU T4 x2**, Internet **On** (pip and the model downloads need it), visibility **Private**. Cells run top to bottom, each explained by the markdown above it; "cell 7" always means the section headed `7.`, not a Jupyter index. Rough cost for the export's default size, on T4s: cell 4 about 10 minutes, cell 5 about 1.5 hours on one GPU, cell 7 several hours. These are estimates from upstream's numbers and FLOP arithmetic, not measurements: time the first minutes of each and stop if it is not what you budgeted.

**Status.** Written against the source of `laya` 0.3.28, `FlagEmbedding` 1.4.2 and `transformers` 5.19.0, and exercised on CPU with tiny stand-in models; it has not yet been run on Kaggle's GPUs. Expect to fix small things on the first run, and delete this paragraph once you have run it.

**Licence.** Cells 6-8 adapt code from `NandhaKishorM/laya` (`notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb`, Apache-2.0); each of them says what was changed. Everything else is this project's.

## 0. Privacy first: nothing in this notebook uploads anything

The dataset attached to this notebook (`pairs.jsonl`) is **verbatim text from your vault**: lecture notes, textbook pages, homework. Treat it like the vault itself.

* The dataset must be **PRIVATE** on Kaggle. The `kaggle datasets create` command is private by default (never pass `--public`); in the web UI, check the visibility switch. Keep **this notebook private** too: its saved outputs include passage excerpts, and the checkpoint is trained on them.
* **Delete the dataset after training**, and the notebook's saved versions with it. The manual (`docs/laya-finetune.md`, section 6) has the commands.
* **Internet has to be On** for `pip` and the three model downloads. That also means any code in a notebook *could* send the attached data anywhere. The code cells below only read the dataset and write under `/tmp` and `/kaggle/working`, and `tests/test_laya_notebook.py` fails if a code cell contains an upload call or starts a program other than `nvidia-smi`, `pip` and `torchrun`. If you edit the notebook, you own that guarantee.
* Never put a Hugging Face **write** token, or any other secret, in Kaggle Secrets for this notebook. Upstream's notebook ends with a cell that publishes the model to a **public** Hugging Face repository. It is deliberately not here.

Nothing runs in this cell. It is the contract for the rest of the notebook.

## 1. Environment check

**What it does:** prints `nvidia-smi` and the torch/CUDA build, asserts that as many GPUs are visible as `NPROC`, and defines the settings every later cell reads: the paths, `NPROC`, the seed, and the `run()` helper.

**Why:** cell 7 trains data-parallel, one process per GPU. A session that was given one GPU would otherwise fail hours in, after the slow cells 4 and 5. `run()` exists because a bare `!command` in a notebook ignores the exit status, so a crashed `torchrun` would sail on into the evaluation; this one raises.

**On one GPU** (a Kaggle session with a single T4, or Colab's free T4): set `NPROC = 1`. Cell 7 then trains in one process and accumulates more micro-batches per update so the effective batch stays 64; expect about twice the wall-clock. Colab has no `/kaggle/...` paths, so change `INPUT_ROOT`, `WORK_DIR` and `SCRATCH` as well.

**Where files go:** vault text (the rows, the tokenised items, the rolling checkpoint) is written to `SCRATCH` (under `/tmp`), and only cell 9 writes to `WORK_DIR`, so a saved version of the notebook should keep just the checkpoint and the zip. Check the Output tab after a saved run to confirm that is what your session keeps; Kaggle's documentation is the authority on that, not this comment.

In [ ]:
# Cell 1 - environment check, and the settings every later cell shares.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # inherited by the torchrun processes in cell 7

import subprocess
import time

INPUT_ROOT = "/kaggle/input"                  # where Kaggle mounts attached datasets
WORK_DIR = "/kaggle/working"                  # saved with the notebook: only cell 9 writes here
SCRATCH = "/tmp/noetrix-laya"                 # vault text and the rolling checkpoint live here, not in the saved output
CKPT_DIR = f"{SCRATCH}/noetrix-laya-ckpt"     # cell 7 trains into this, cell 8 evaluates it, cell 9 ships it
OUT_DIR = f"{WORK_DIR}/noetrix-laya"
NPROC = 2                                     # one training process per GPU; set 1 on a single-GPU runtime
SEED = 0
RUN = {"seed": SEED}                          # provenance, filled in by the later cells and written to metrics.json


def run(cmd):
    """Run a command, stream its output into the cell, and raise if it fails (a bare `!cmd` ignores the exit status)."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"`{' '.join(cmd)[:120]}` exited with status {proc.returncode}")


run(["nvidia-smi"])

import torch

n_gpu = torch.cuda.device_count()
print(f"torch {torch.__version__} | CUDA build {torch.version.cuda} | GPUs visible: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB  compute capability {p.major}.{p.minor}")
assert n_gpu >= NPROC, (
    f"NPROC = {NPROC} but torch sees {n_gpu} GPU(s). On Kaggle choose Accelerator = GPU T4 x2 in the notebook options; "
    "on a single-GPU runtime set NPROC = 1 above and cell 7 will train in one process.")
os.makedirs(SCRATCH, exist_ok=True)

## 2. Pinned installs

**What it does:** installs four packages at exact versions, imports them, and prints what is now installed, so that a broken install fails here rather than after the doc2query run.

| Pin | Why this version |
|---|---|
| `laya==0.3.28` | Laya was weeks old when this was written and shipped dozens of releases, so its API moves; a floating install is a different program next week. 0.3.28 is the release whose `laya.load`, `predict_batch`, `laya.train` and `laya.common` this notebook was written against (read from the wheel, which is byte-identical to the upstream repository at that release). |
| `transformers==5.19.0` | The version `laya` 0.3.28 was exercised on end to end (load, predict, batching), and the one the RAG's own venv already resolves to. `laya` itself only needs `>=4.48.0`; its README says its stack resolves to transformers 5.x. |
| `sentence-transformers==5.5.1` | The version pinned in the repo's `requirements.txt`, so the MiniLM baseline in cell 8 is scored by the same `CrossEncoder` code as the RAG's own reranker. |
| `FlagEmbedding==1.4.2` | The teacher's runtime (`FlagReranker`). The latest release when this was written; it declares `transformers>=4.44.2,<6` and ships v4/v5 tokenizer compatibility helpers, which its pair tokenisation needs under transformers 5. |

These four resolve together with `pip`'s resolver on Linux / Python 3.11 (checked with `uv pip compile`, nothing installed).

`torch` is not pinned and not touched. Kaggle's image carries the CUDA build that matches its driver, and upgrading it with `pip` is how a working GPU session becomes a broken one. None of the four declares a newer `torch` than that: `laya` needs `torch>=2.0`, `sentence-transformers` `>=1.11`, `FlagEmbedding` `>=1.6`, and `transformers` declares none. If the import check below fails complaining about `torch`, the image's `torch` is too old for these pins: do not upgrade it blindly, work out which pin to move.

In [ ]:
# Cell 2 - pinned installs. torch is deliberately not in the list: Kaggle's image carries the CUDA build that matches its driver.
import sys
from importlib.metadata import version

PINS = ["laya==0.3.28", "transformers==5.19.0", "sentence-transformers==5.5.1", "FlagEmbedding==1.4.2"]
run([sys.executable, "-m", "pip", "install", "-q", *PINS])

import FlagEmbedding            # imported now so a broken install fails here, not after the doc2query run
import laya
import sentence_transformers
import transformers

RUN["versions"] = {name: version(name) for name in ("laya", "transformers", "sentence-transformers", "FlagEmbedding", "torch")}
for name, v in RUN["versions"].items():
    print(f"{name:24s}{v}")
assert laya.__version__ == "0.3.28" and transformers.__version__ == "5.19.0", (
    "a different version was imported before the install finished: restart the session (Run > Restart) and run again from cell 1")

## 3. Load the pairs

**What it does:** finds `pairs.jsonl` under `/kaggle/input`, checks every row against the export tool's format (`{pid, passage, negatives: [{id, text}], meta: {source_file, file_type}}`), and prints the counts so you can see what you are about to train on.

**Why:** the attached dataset is the only thing that carries vault text into this notebook, and its mount path is `/kaggle/input/<slug>/`, or `/kaggle/input/<owner>/<slug>/` when names clash (Kaggle's CLI documentation), so the cell searches for the file instead of hard-coding a path, and refuses to go on if it finds none or several. A malformed row stops the run here, with its line number, instead of surfacing as a tokenizer error in cell 6. A passage with no negatives carries no ranking information, so it is dropped and counted.

The export already left out every chunk of every gold file in the eval sets, so the questions you will eventually be scored on were not trained on; this cell does not re-check that.

In [ ]:
# Cell 3 - load pairs.jsonl from the attached private dataset.
import json
from collections import Counter
from pathlib import Path

import numpy as np

matches = sorted(Path(INPUT_ROOT).rglob("pairs.jsonl"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"expected exactly one pairs.jsonl under {INPUT_ROOT}, found {[str(m) for m in matches]}. "
        "Attach the private dataset to this notebook (Add Input) and make sure it holds a single pairs.jsonl.")
PAIRS_PATH = matches[0]

records, no_negatives = [], 0
with open(PAIRS_PATH, encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        row = json.loads(line)
        problems = [k for k in ("pid", "passage", "negatives", "meta") if k not in row]
        if not problems and not (isinstance(row["passage"], str) and row["passage"].strip()):
            problems.append("passage is empty")
        if not problems and not all(isinstance(n, dict) and n.get("text") for n in row["negatives"]):
            problems.append("a negative has no text")
        if problems:
            raise ValueError(f"{PAIRS_PATH}:{line_no} is not a row of the export format: {problems}")
        if row["negatives"]:
            records.append(row)
        else:
            no_negatives += 1

if not records:
    raise ValueError(f"{PAIRS_PATH} holds no passage with at least one negative")
if len({r["pid"] for r in records}) != len(records):
    raise ValueError("duplicate pid values in pairs.jsonl: the export tool never writes these")

n_neg = [len(r["negatives"]) for r in records]
print(f"{len(records)} passages ({no_negatives} dropped for having no negatives) from {PAIRS_PATH}")
print(f"negatives per passage: min {min(n_neg)}, median {int(np.median(n_neg))}, max {max(n_neg)}")
print("file types:", dict(Counter(r["meta"].get("file_type", "?") for r in records)))
RUN["data"] = {"passages": len(records), "dropped_without_negatives": no_negatives}

## 4. doc2query: one question per passage

**What it does:** asks `Qwen/Qwen2.5-1.5B-Instruct` (fp16, greedy) to write one standalone question that each passage answers, cleans the replies, drops the unusable ones, and prints 20 passages with their questions to eyeball.

**Why a generator, and why this one.** The passages are the positives, and Laya and the teacher need a question to judge them against; the vault has none. Qwen2.5-1.5B-Instruct (Apache-2.0 per the Qwen2.5 repository, ungated, about 3 GB in fp16) is an instruct model, so "write ONE standalone question, never mention the passage" is plain prompting, and it copes with the corpus mix of lecture prose, textbook text, code and LaTeX. The classic alternative, a doc2query T5 trained on MS MARCO, learned from roughly 60-word web snippets and tends to write short, generic, web-search-style queries that mirror the surface text and degrade on code and long chunks; T5 models are also widely reported to overflow in fp16, which on a T4 (no bf16) means fp32 and slower. This choice is reasoned, not benchmarked: the spot-check printed below is how you find out if it is wrong.

**Greedy is temperature 0:** `do_sample=False`, one beam, and the sampling defaults Qwen ships (temperature 0.7, top-p 0.8, top-k 20) are cleared so nothing silently depends on them. The one shipped knob left in force is Qwen's repetition penalty: it is deterministic, and it keeps a 1.5B model from looping. Everything is seeded as well. The model's commit hash is printed; paste it into `GEN_REVISION` to pin later runs (it could not be looked up when this was written).

**Cleaning rules:** first line only; strip numbering, bullets and a leading "Question:"; reject anything that mentions "the passage", "the text", "this document" or "above" (the label would then be true only for that one passage), anything under 3 words, and anything with neither a question mark nor 6 or more words. A question two passages share cannot single either of them out, so both are dropped. If more than `MAX_DROP_FRACTION` of the replies are unusable the cell raises: garbage usually means fp16 overflowed (set `GEN_DTYPE = torch.float32`), prose means the prompt needs tightening.

The spot-check prints passage text, so keep the notebook private (cell 0).

In [ ]:
# Cell 4 - doc2query: one standalone question per passage, greedy decoding.
import gc
import random
import re
from collections import Counter

from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
GEN_REVISION = None                  # paste the commit hash printed below here to pin the next run
GEN_DTYPE = torch.float16            # the T4 has no bf16; if the guard below trips on garbage output, try torch.float32
GEN_BATCH = 32
GEN_MAX_NEW_TOKENS = 48
GEN_PASSAGE_TOKENS = 1024            # a longer passage is cut here before it is put in the prompt
MAX_DROP_FRACTION = 0.5              # stop, rather than train on the remainder, if more questions than this are unusable
MIN_WORDS = 3
SYSTEM = "You write search questions for a personal note archive."
PROMPT = (
    "Write ONE standalone question that the passage below answers. The question must make sense to someone who has "
    "not seen the passage, so never mention 'the passage', 'the text' or 'this document'. "
    "Reply with the question only.\n\nPassage:\n{passage}"
)
BANNED = re.compile(r"\b(passage|the text|this document|above)\b", re.IGNORECASE)
LEAD_IN = re.compile(r"^\s*(?:(?:question|query)\s*[:\-]\s*|q\s*:\s*|\d+[.)]\s*|[-*\u2022]\s+)", re.IGNORECASE)


# Pure helpers (clean_query is unit-tested by tests/test_laya_notebook.py).
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clean_query(raw):
    """The first line of a model reply as a usable question, or None. Rejected: replies that point back at the
    passage (the label would only be true for that one passage), one- and two-word fragments, and statements
    that are neither a question nor a sentence."""
    lines = raw.strip().splitlines()
    if not lines:
        return None
    q = LEAD_IN.sub("", lines[0]).strip().strip("\"'\u201c\u201d`").strip()
    words = len(q.split())
    if BANNED.search(q) or words < MIN_WORDS or ("?" not in q and words < 6):
        return None
    return q


seed_everything(SEED)
tok_g = AutoTokenizer.from_pretrained(GEN_ID, revision=GEN_REVISION)
tok_g.padding_side = "left"          # a decoder-only model must be left-padded for batched generate()
gen = AutoModelForCausalLM.from_pretrained(GEN_ID, revision=GEN_REVISION, dtype=GEN_DTYPE).to("cuda:0").eval()
GEN_COMMIT = getattr(gen.config, "_commit_hash", None)
print(f"generator {GEN_ID} @ {GEN_COMMIT}  (set GEN_REVISION to that hash to pin it)")


def build_prompt(passage):
    ids = tok_g(passage, add_special_tokens=False, truncation=True, max_length=GEN_PASSAGE_TOKENS)["input_ids"]
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": PROMPT.format(passage=tok_g.decode(ids))}]
    return tok_g.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def generate_queries(passages):
    order = sorted(range(len(passages)), key=lambda i: -len(passages[i]))     # longest first: little padding, and an out-of-memory error shows in batch 1
    raw, t0 = [None] * len(passages), time.time()
    for n, start in enumerate(range(0, len(order), GEN_BATCH)):
        idx = order[start:start + GEN_BATCH]
        enc = tok_g([build_prompt(passages[i]) for i in idx], return_tensors="pt", padding=True,
                    add_special_tokens=False).to("cuda:0")
        # temperature 0 = greedy: sampling is off, and the sampling knobs Qwen ships in its generation_config are cleared
        out = gen.generate(**enc, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                           temperature=None, top_p=None, top_k=None, pad_token_id=tok_g.pad_token_id)
        for i, text in zip(idx, tok_g.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            raw[i] = text
        if n % 20 == 0:
            print(f"  {min(start + GEN_BATCH, len(order))}/{len(order)} passages, {time.time() - t0:.0f}s")
    return raw


raw = generate_queries([r["passage"] for r in records])
queries = [clean_query(t) for t in raw]
counts = Counter(q.casefold() for q in queries if q)
# a question two passages share cannot single either of them out, so both are dropped
queried = [dict(r, query=q) for r, q in zip(records, queries) if q and counts[q.casefold()] == 1]
n_unusable = sum(1 for q in queries if q is None)
print(f"{len(queried)} usable questions; dropped {n_unusable} unusable and {len(records) - n_unusable - len(queried)} shared")
if 1 - len(queried) / len(records) > MAX_DROP_FRACTION:
    raise RuntimeError(
        f"{1 - len(queried) / len(records):.0%} of the generated questions are unusable (limit {MAX_DROP_FRACTION:.0%}). "
        "Look at the raw replies, e.g. print(raw[:5]): garbage means fp16 overflowed (set GEN_DTYPE = torch.float32), "
        "prose means the prompt needs tightening.")

print("\nSpot-check these before the GPU-hours cells (the passage is cut to 160 characters):")
for r in random.Random(SEED).sample(queried, min(20, len(queried))):
    print(f"- {r['query']}\n    <- {r['passage'][:160]!r}")

RUN["generator"] = {"id": GEN_ID, "commit": GEN_COMMIT, "dtype": str(GEN_DTYPE), "do_sample": False,
                    "max_new_tokens": GEN_MAX_NEW_TOKENS, "queries_kept": len(queried), "unusable": n_unusable}
del gen
gc.collect()
torch.cuda.empty_cache()

## 5. Teacher labels

**What it does:** scores `(question, candidate)` for the passage and for every hard negative with `BAAI/bge-reranker-v2-m3` (`FlagReranker(use_fp16=True)`, `compute_score(..., normalize=True)`, which is the sigmoid of the logit), so each candidate gets a probability `p` of being relevant. It scores a random pilot of 200 passages first, and stops if the teacher often prefers another candidate to the passage the question was written from.

**Why a teacher, and why soft labels.** The vault has no relevance labels (the eval questions are held out for scoring), so the labels have to come from a stronger model. Hard negatives are a passage's nearest neighbours in other files, and some of them genuinely answer the question; the teacher's probability says so, where a 0/1 label would train the student to be confidently wrong. `devices="cuda:0"` is explicit because FlagEmbedding's default means "every visible GPU", which starts a multi-process pool. The model's commit hash is printed; `FlagReranker` takes no revision argument, so to pin the teacher in a later run, `snapshot_download` that commit and pass the local directory as `TEACHER_ID`.

**Why the pilot.** If the questions are too generic, the teacher's top candidate is some other passage, the labels are noise, and the whole GPU run is wasted. A rule of thumb: if the source passage is not the teacher's top-1 for more than about 40% of pilot questions, fix the prompt in cell 4. The cell raises above `MAX_TOP1_MISS`.

**Cost (estimates, not measurements).** A third party measured about 33 pairs/s for this model on an RTX A5000; a T4 is roughly 2.5 to 3 times slower, so about 10 to 14 pairs/s, which is on the order of 1.5 hours for the export's default size on one T4. The cell prints the measured rate after the first 4,096 pairs: stop and re-export a smaller set if it is not what you budgeted.

The student can only imitate these numbers. Its calibration is bounded by the teacher's, and it cannot beat the teacher on the teacher's own labels.

In [ ]:
# Cell 5 - teacher labels: P(relevant) from bge-reranker-v2-m3 for the passage and for every negative.
from FlagEmbedding import FlagReranker

TEACHER_ID = "BAAI/bge-reranker-v2-m3"
TEACHER_BATCH = 32
TEACHER_CHUNK = 4096         # pairs per compute_score call: visible progress, and the speed is known early
PILOT_PASSAGES = 200         # scored first, so a bad doc2query prompt is found before the long run
MAX_TOP1_MISS = 0.40         # rule of thumb: above this, fix the prompt in cell 4

# devices must be explicit: the default (None) means every visible GPU, which starts a multi-process pool
teacher = FlagReranker(TEACHER_ID, use_fp16=True, devices="cuda:0")
TEACHER_COMMIT = getattr(teacher.model.config, "_commit_hash", None)
print(f"teacher {TEACHER_ID} @ {TEACHER_COMMIT}")


def candidate_texts(record):
    return [record["passage"]] + [n["text"] for n in record["negatives"]]


def score_records(indices):
    """{index: [P(passage), P(negative 1), ...]}: the teacher's sigmoid(logit) for (query, candidate)."""
    pairs = [(queried[i]["query"], t) for i in indices for t in candidate_texts(queried[i])]
    flat, t0 = [], time.time()
    for start in range(0, len(pairs), TEACHER_CHUNK):
        flat += teacher.compute_score(pairs[start:start + TEACHER_CHUNK], normalize=True, batch_size=TEACHER_BATCH)
        rate = len(flat) / (time.time() - t0)
        print(f"  {len(flat)}/{len(pairs)} pairs  {rate:.1f} pairs/s  ~{(len(pairs) - len(flat)) / rate / 60:.0f} min left")
    out, k = {}, 0
    for i in indices:
        n = 1 + len(queried[i]["negatives"])
        out[i] = flat[k:k + n]
        k += n
    return out


def top1_miss(scores):
    """Share of queries whose own passage is NOT the teacher's top candidate (a tie counts as top)."""
    return float(np.mean([s[0] < max(s[1:]) for s in scores.values()]))


pilot = random.Random(SEED).sample(range(len(queried)), min(PILOT_PASSAGES, len(queried)))
P = score_records(pilot)
miss = top1_miss(P)
print(f"pilot: the source passage is not the teacher's top-1 for {miss:.0%} of {len(P)} queries (limit {MAX_TOP1_MISS:.0%})")
if miss > MAX_TOP1_MISS:
    raise RuntimeError(
        "the teacher often prefers another candidate to the passage the question was written from: the questions are too "
        "generic or too far from the passage. Tighten PROMPT in cell 4 and rerun cells 4-5 before spending the long run.")
P.update(score_records([i for i in range(len(queried)) if i not in P]))
scored = [dict(r, p=P[i]) for i, r in enumerate(queried)]

flat_pos = np.array([s["p"][0] for s in scored])
flat_neg = np.array([x for s in scored for x in s["p"][1:]])
print(f"teacher P(relevant): passages mean {flat_pos.mean():.2f}; negatives mean {flat_neg.mean():.3f}, "
      f"{(flat_neg >= 0.5).mean():.1%} of negatives score >= 0.5 (soft labels keep those; hard labels would not)")
print(f"whole run: the source passage is not the teacher's top-1 for {top1_miss(dict(enumerate(s['p'] for s in scored))):.0%}")
RUN["teacher"] = {"id": TEACHER_ID, "commit": TEACHER_COMMIT, "fp16": True, "normalize": True,
                  "pilot_top1_miss": miss, "mean_p_passage": float(flat_pos.mean()), "mean_p_negative": float(flat_neg.mean())}
del teacher
gc.collect()
torch.cuda.empty_cache()

## 6. Training rows

**What it does:** turns every scored candidate into one Laya training row, splits 90/10 by passage, tokenises the training rows for cell 7, and keeps the held-out passages as ranked groups for cell 8.

**The row format** is upstream's `{state, questions, gold}`. ("RLCD" names the training objective, not a file format.) The candidate text is the *state*; the question is a `noul` (yes/no) question that carries the generated query; the gold is the teacher's probability, `{"false": 1 - p, "true": p}`, plus a `"true"`/`"false"` label. They are native JSON objects, not JSON-in-strings (upstream's benchmark dataset stores strings and `json.loads`es them). The wording, `Does the passage answer: <query>?` with the criteria `Irrelevant.` and `Answers the query.`, is defined once (`make_question`) and reused by cell 8. The inference adapter in the repo uses the identical text, because the question is part of the model's input: training on one prompt and serving another quietly breaks the model. A query that already ends in "?" yields "??" in both places, consistently.

**The split is by passage.** A passage and its negatives share one pid, so they land on the same side. Beyond that, no training row's text may equal a held-out passage (a held-out positive that turned up in training as somebody else's negative would not be held out); those rows are dropped and counted. Passages from the same *file* can still fall on both sides, so the held-out numbers flatter the student a little compared with truly unseen files.

**Held-out candidates are shuffled per passage** with a fixed seed. Laya rounds P(true) to 4 decimals, so ties near 0 and 1 are common, and a tie is resolved by candidate order; with the positive always first, every tie would favour it.

**Tokenisation** goes through `laya.train.items_from_rows`, which renders each question exactly as `laya.load(...).predict_batch` will at inference. Rows that cannot become items are counted by reason, and the cell raises if more than 1% are. `MAX_LEN, HEAD_MAX_LEN = 512, 192` is the base English checkpoint's own budget, stated once here and written into the saved config in cell 7. The base checkpoint is downloaded as just the four paths `laya.load` reads, not the whole repository, at the revision `laya` itself lists as reviewed (`laya.revisions.PINNED_REVISIONS`). That commit could not be looked up when this was written; if the download fails with a revision error, drop `revision=` only after deciding that an unpinned base is acceptable.

In [ ]:
# Cell 6 - Laya training rows ({state, questions, gold}), split 90/10 by passage, tokenised for cell 7.
# The tokenisation step adapts cell 6 of NandhaKishorM/laya notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb (Apache-2.0)
# to this task's rows; the upstream cell reads a Hugging Face dataset and calls build_sequence() directly, this one goes
# through laya.train.items_from_rows, which renders each question exactly as inference will.
import math
import os

from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.revisions import PINNED_REVISIONS
from laya.train import encode_item, items_from_rows
from transformers import AutoTokenizer

BASE_ID = "convaiinnovations/laya"
MAX_LEN, HEAD_MAX_LEN = 512, 192     # sequence budget: the base English checkpoint's own values, written into the saved config too
HELDOUT_FRAC = 0.10
MAX_SKIPPED_FRACTION = 0.01


# Pure helpers: unit-tested by tests/test_laya_notebook.py, which pulls them out of this notebook and runs them.
def make_question(query):
    """The one question every row asks. The inference adapter in the repo uses the identical wording."""
    return {"rel": {"type": "noul", "instructions": f"Does the passage answer: {query}?",
                    "criteria": {"false": "Irrelevant.", "true": "Answers the query."}}}


def make_row(text, query, p):
    """One training row: the candidate text is the state, the teacher's probability is the soft label."""
    if not 0.0 <= p <= 1.0:
        raise ValueError(f"teacher probability {p!r} is not in [0, 1]")
    return {"state": text, "questions": make_question(query),
            "gold": {"rel": {"probabilities": {"false": 1.0 - p, "true": p}, "label": "true" if p >= 0.5 else "false"}}}


def split_by_passage(pids, heldout_frac, seed):
    """(train pids, held-out pids). A passage and its negatives share one pid, so they can never straddle the split."""
    ordered = sorted(set(pids))
    if len(ordered) < 2:
        raise ValueError("need at least two passages to hold one out")
    random.Random(seed).shuffle(ordered)
    n_held = max(1, round(len(ordered) * heldout_frac))
    return set(ordered[n_held:]), set(ordered[:n_held])


def build_train_rows(scored, train_pids, heldout_texts):
    """(rows, n_shared). Rows for the training passages, minus any candidate whose text is a held-out passage: the
    held-out positive is never seen in training, in any role."""
    rows, shared = [], 0
    for r in scored:
        if r["pid"] not in train_pids:
            continue
        for text, p in zip([r["passage"]] + [n["text"] for n in r["negatives"]], r["p"]):
            if text in heldout_texts:
                shared += 1
            else:
                rows.append(make_row(text, r["query"], p))
    return rows, shared


def build_heldout(scored, heldout_pids, seed):
    """One group per held-out passage. Candidates are shuffled per passage so that equal scores (Laya rounds P(true) to
    4 decimals) cannot favour the positive by sitting first."""
    groups = []
    for r in scored:
        if r["pid"] not in heldout_pids:
            continue
        texts = [r["passage"]] + [n["text"] for n in r["negatives"]]
        order = list(range(len(texts)))
        random.Random(f"{seed}:{r['pid']}").shuffle(order)
        groups.append({"pid": r["pid"], "query": r["query"], "texts": [texts[i] for i in order],
                       "label": [int(i == 0) for i in order], "teacher_p": [r["p"][i] for i in order]})
    return groups


train_pids, heldout_pids = split_by_passage([r["pid"] for r in scored], HELDOUT_FRAC, SEED)
heldout_texts = {r["passage"] for r in scored if r["pid"] in heldout_pids}
train_rows, n_shared = build_train_rows(scored, train_pids, heldout_texts)
heldout = build_heldout(scored, heldout_pids, SEED)
with open(f"{SCRATCH}/rows_train.jsonl", "w", encoding="utf-8") as f:
    for row in train_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
labels = np.array([row["gold"]["rel"]["probabilities"]["true"] for row in train_rows])
print(f"{len(train_pids)} training passages -> {len(train_rows)} rows; {len(heldout_pids)} held-out passages "
      f"-> {len(heldout)} groups of {len(heldout[0]['texts'])} candidates")
print(f"{n_shared} candidate rows dropped because their text is a held-out passage; teacher P(true) >= 0.5 in {(labels >= 0.5).mean():.1%} of rows")

# The base checkpoint: only the four things laya.load reads, at the revision laya itself lists as reviewed.
base_dir = snapshot_download(BASE_ID, revision=PINNED_REVISIONS[BASE_ID],
                             allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"])
_fix_tokenizer_config(base_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(base_dir, "tokenizer"))
with open(os.path.join(base_dir, "rl_agent_config.json"), encoding="utf-8") as f:
    base_cfg = json.load(f)
print(f"base {BASE_ID} @ {PINNED_REVISIONS[BASE_ID][:12]}: max_len {base_cfg.get('max_len')}, head_max_len {base_cfg.get('head_max_len')}"
      f" -> training and saving at {MAX_LEN} / {HEAD_MAX_LEN}")

items, skipped = items_from_rows(tok, train_rows, MAX_LEN, HEAD_MAX_LEN)
if sum(skipped.values()) > MAX_SKIPPED_FRACTION * len(train_rows):
    raise RuntimeError(f"{sum(skipped.values())} of {len(train_rows)} rows could not be tokenised: {skipped}")
encoded = [encode_item(tok, it, MAX_LEN, HEAD_MAX_LEN) for it in items]
torch.save(encoded, f"{SCRATCH}/train_items.pt")
print(f"{len(encoded)} items tokenised (skipped: {skipped or 'none'}), saved to {SCRATCH}/train_items.pt")
RUN["rows"] = {"train_passages": len(train_pids), "heldout_passages": len(heldout_pids), "train_rows": len(train_rows),
               "items": len(encoded), "skipped": skipped, "dropped_shared_with_heldout": n_shared,
               "max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN, "base": {"id": BASE_ID, "revision": PINNED_REVISIONS[BASE_ID]}}

## 7. Fine-tune

**What it does:** writes the recipe (`TRAIN_CFG`) and the training script to `SCRATCH`, runs `torchrun --standalone --nproc_per_node=NPROC` on it, and checks that the final directory is one `laya.load` can open.

**The recipe is upstream's DDP recipe** (its Kaggle notebook and `docs/finetune.md`): 4 epochs; micro-batch 8 per GPU; effective batch 64 (8 per micro-batch x 2 GPUs x 4 accumulation steps; the script derives the accumulation from the process count, so a single process accumulates 8); AdamW with weight decay 0.01; learning rate 2.5e-5 for the encoder and 1e-4 for the head, cosine decay to 1e-6 with no warm-up; gradient clip 1.0; fp16 autocast with a GradScaler and gradient checkpointing. The objective is a policy-gradient term (4 noisy samples per item, sigma annealed from 0.4 to 0.1, spherical weight 0.75, ranked-probability weight 1.0) plus soft cross-entropy against the teacher's distribution. A calibration slice of min(400, 10%) rows is held out before training, and a temperature is fitted on it afterwards.

**Where this differs from upstream's script, and why:**

1. **`max_len` 512 / `head_max_len` 192 are forced into every saved config.** Upstream writes 1024 / 256 into the config while its items were tokenised at 512 / 192, so its checkpoint advertises a budget it was not trained at.
2. **The final directory is written by `laya.train.save_checkpoint`** (weights, encoder config, tokenizer, then `rl_agent_config.json` last), and the cell checks all four entries afterwards. Upstream's rolling `checkpoint_latest/` has *no* `rl_agent_config.json`, so `laya.load` refuses it. Here the rolling copy goes through the same writer and lives in `SCRATCH`: a crashed run leaves something loadable (with the base checkpoint's temperature, not a fitted one), and the shipped directory holds only what the model needs.
3. **Nothing is swallowed.** Upstream falls back to a temperature of 1.2 if the fit raises, and `!torchrun` ignores a crash. Here both raise.
4. **`reference_compile` is switched off**, as `laya.load` does at inference; upstream's own T4 benchmark notebook notes that `torch.compile` hurts on T4-class GPUs.
5. The objective, schedule helpers, temperature fit and checkpoint writer are `laya`'s own functions (`laya.train`, `laya.calibrate`), not copies. The process group, the sharding and the DDP wrapper are the notebook's.

**Time it before you commit.** The log prints the loss, the logit spread and an ETA every 50 micro-batches. Estimates for the export's default size, not measurements: roughly 3 to 5 hours on 2 T4s by a FLOP estimate, roughly 9 hours at the rate upstream's README implies. Sessions are capped at 12 hours; if the ETA is too long, stop, lower `epochs`, or re-export with fewer negatives.

**Collapse risk (upstream issue 963).** On small datasets upstream's 4-epoch default can collapse to a constant: the report has accuracy at chance and a mean within-row logit range of 0.01 against 0.62 for the untuned checkpoint, and 6 of 15 identical runs never left the plateau. A full-size run has roughly 50 times more optimizer updates than that report, but a small spike run is close to the regime where it happened, so a collapsed spike is not evidence against the idea. **How to spot it:** in the log, `logit spread` (the mean absolute gap between the true and false logits) should climb over the first epoch into whole units while the loss keeps falling. A spread that stays near 0.01 to 0.1 on a flat loss is a collapse: a binary `noul` model then answers one number, near 0.5 or near the label prior, for everything. Cell 8 repeats the check on held-out data and raises if the student's spread is under a fifth of the teacher's.

In [ ]:
# Cell 7 - fine-tune with upstream's DDP recipe. The process-group scaffolding below adapts train_ddp.py from
# NandhaKishorM/laya notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb (Apache-2.0); the objective, the schedule
# helpers, the temperature fit and the checkpoint writer are laya's own (laya.train / laya.calibrate), not copies.
import shutil

TRAIN_CFG = dict(
    epochs=4, micro_batch=8, effective_batch=64,                    # 64 = 8 per micro-batch x processes x accumulation steps
    encoder_lr=2.5e-5, head_lr=1e-4, weight_decay=0.01,             # AdamW; cosine decay to min_lr, no warm-up
    min_lr=1e-6, grad_clip=1.0,
    rl_samples=4, sigma_start=0.4, sigma_end=0.1,                   # the policy-gradient half of the objective
    w_sph=0.75, w_rps=1.0,                                          # proper-scoring-rule reward weights
    calib_max=400, calib_frac=0.1, calib_seed=20260922,             # slice held out BEFORE training to fit the temperature
    max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN, seed=SEED,
)

TRAIN_SCRIPT = r'''
import json
import math
import os
import random
import sys
import time

import torch
import torch.distributed as dist
from safetensors.torch import load_file
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoTokenizer

from laya.calibrate import fit_temperature_map
from laya.common import build_model, collate_items
from laya.train import rlcd_loss, save_checkpoint, sigma_at, split_calibration

BASE_DIR, CKPT_DIR, SCRATCH = sys.argv[1:4]
with open(os.path.join(SCRATCH, "train_cfg.json")) as f:
    H = json.load(f)


def forward(model, batch, device):
    """One forward pass under fp16 autocast: (float32 logits, act-head output)."""
    with torch.autocast("cuda", dtype=torch.float16):
        logits, act = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                            batch["marker_pos"].to(device), batch["marker_mask"].to(device), batch["qtype"].to(device))
    return logits.float(), act


def main():
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    local = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local)
    device = torch.device("cuda", local)
    torch.manual_seed(H["seed"] + rank)
    per_update = H["micro_batch"] * world
    if H["effective_batch"] % per_update:
        raise ValueError(f"effective_batch {H['effective_batch']} is not a multiple of micro_batch x processes = {per_update}")
    grad_accum = H["effective_batch"] // per_update
    budget = dict(max_len=H["max_len"], head_max_len=H["head_max_len"])    # forced into every config this script saves

    with open(os.path.join(BASE_DIR, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    tok = AutoTokenizer.from_pretrained(os.path.join(BASE_DIR, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(BASE_DIR, "encoder"))
    model.load_state_dict(load_file(os.path.join(BASE_DIR, "model.safetensors")), strict=True)
    model.encoder.config.reference_compile = False    # what laya.load does; torch.compile only slows a T4 down
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device).train()
    ddp = DDP(model, device_ids=[local], find_unused_parameters=True)

    items = torch.load(os.path.join(SCRATCH, "train_items.pt"), weights_only=False)
    train_items, calib_items = split_calibration(items, H["calib_max"], H["calib_frac"], H["calib_seed"])
    train_items = train_items[: len(train_items) // world * world]    # every rank must run the same number of steps (laya issue 678)
    mine = train_items[rank::world]

    micro = H["micro_batch"]
    micro_per_epoch = math.ceil(len(mine) / micro)
    updates = math.ceil(micro_per_epoch / grad_accum) * H["epochs"]
    enc = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
    head = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": enc, "lr": H["encoder_lr"]}, {"params": head, "lr": H["head_lr"]}],
                            weight_decay=H["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, updates), eta_min=H["min_lr"])
    scaler = torch.amp.GradScaler("cuda")
    if rank == 0:
        print(f"{len(train_items)} train items ({len(calib_items)} held out to fit the temperature), {len(mine)} per process, "
              f"{world} process(es), grad_accum {grad_accum}, {updates} optimizer updates", flush=True)

    t0, done, total = time.time(), 0, micro_per_epoch * H["epochs"]
    for epoch in range(H["epochs"]):
        random.Random(H["seed"] + epoch + rank).shuffle(mine)
        sigma = sigma_at(epoch, H["epochs"], H["sigma_start"], H["sigma_end"])
        opt.zero_grad(set_to_none=True)
        loss_sum = spread_sum = 0.0
        n = 0
        for start in range(0, len(mine), micro):
            batch = collate_items([mine[start:start + micro]], tok.pad_token_id)
            logits, act = forward(ddp, batch, device)
            loss = rlcd_loss(logits, batch["target"].to(device), batch["marker_mask"].to(device), batch["qtype"].to(device),
                             sigma, H["rl_samples"], H["w_sph"], H["w_rps"])
            scaler.scale(loss / grad_accum + 0.0 * act.sum()).backward()    # 0.0 * act keeps the unused act head in the DDP graph
            n += 1
            done += 1
            if n % grad_accum == 0 or start + micro >= len(mine):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ddp.parameters(), H["grad_clip"])
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
            loss_sum += loss.item()
            spread_sum += (logits[:, 1] - logits[:, 0]).abs().mean().item()    # |logit(true) - logit(false)|: ~0 means collapsed
            if rank == 0 and n % 50 == 0:
                minutes = (time.time() - t0) / 60
                print(f"  epoch {epoch + 1}/{H['epochs']} micro-batch {n}/{micro_per_epoch} loss {loss.item():.4f} "
                      f"logit spread {spread_sum / n:.3f} lr {sched.get_last_lr()[0]:.2e} "
                      f"{minutes:.1f} min elapsed, ~{minutes / done * (total - done):.0f} min to go", flush=True)
        dist.barrier()
        if rank == 0:
            print(f"=== epoch {epoch + 1}/{H['epochs']}: mean loss {loss_sum / n:.4f}, mean logit spread {spread_sum / n:.3f} ===",
                  flush=True)
            # rolling checkpoint: complete (config included), so a crashed run leaves something laya.load can open
            save_checkpoint(model, tok, dict(cfg, **budget), os.path.join(SCRATCH, "checkpoint_latest"))

    dist.barrier()
    if rank == 0:
        model.eval()
        records = []
        with torch.no_grad():
            for start in range(0, len(calib_items), 16):
                chunk = calib_items[start:start + 16]
                logits, _ = forward(model, collate_items([chunk], tok.pad_token_id), device)
                for row, it in zip(logits.cpu().numpy(), chunk):
                    k = len(it["markers"])
                    records.append((it["qtype"], row[:k], it["target"], k))
        fitted = fit_temperature_map(records)
        out_cfg = dict(cfg, **budget, fine_tuned=True, temperature=fitted["temperature"])
        out_cfg.pop("temperature_by_options", None)    # an inherited bucket map would take precedence and mask the new fit
        if fitted["temperature_by_options"]:
            out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        out_cfg["training"] = dict(out_cfg.get("training") or {}, noetrix=H)
        save_checkpoint(model, tok, out_cfg, CKPT_DIR)    # weights, encoder, tokenizer, then rl_agent_config.json last
        print(f"saved {CKPT_DIR}; temperature (choice, score, noul) = {fitted['temperature']}", flush=True)
    dist.destroy_process_group()


if __name__ == "__main__":
    main()
'''


# Unit-tested by tests/test_laya_notebook.py.
def checkpoint_problem(model_dir):
    """What laya.load would trip over in this directory, or None. All four entries must exist: a missing one is not
    always reported locally (without tokenizer/ and encoder/ laya falls through to a Hugging Face lookup), and
    upstream's rolling checkpoint_latest/ has no rl_agent_config.json at all."""
    d = os.path.abspath(os.path.expanduser(model_dir))
    if not os.path.isdir(d):
        return f"directory {d} does not exist"
    for name, is_dir in (("rl_agent_config.json", False), ("model.safetensors", False), ("tokenizer", True), ("encoder", True)):
        path = os.path.join(d, name)
        if not (os.path.isdir(path) if is_dir else os.path.isfile(path)):
            return f"{d} has no {name}"
    return None


with open(f"{SCRATCH}/train_cfg.json", "w") as f:
    json.dump(TRAIN_CFG, f, indent=2)
with open(f"{SCRATCH}/train_ddp.py", "w") as f:
    f.write(TRAIN_SCRIPT)
if os.path.isdir(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)             # an old checkpoint must never be mistaken for this run's
run(["torchrun", "--standalone", f"--nproc_per_node={NPROC}", f"{SCRATCH}/train_ddp.py", base_dir, CKPT_DIR, SCRATCH])

problem = checkpoint_problem(CKPT_DIR)
if problem:
    raise RuntimeError(f"training finished but the final checkpoint is not loadable: {problem}")
with open(f"{CKPT_DIR}/rl_agent_config.json", encoding="utf-8") as f:
    saved = json.load(f)
print(f"final checkpoint OK: max_len {saved['max_len']}, head_max_len {saved['head_max_len']}, "
      f"temperature {saved['temperature']}, fine_tuned {saved.get('fine_tuned')}")
RUN["training"] = {"recipe": TRAIN_CFG, "nproc": NPROC, "temperature": saved["temperature"]}

## 8. Held-out evaluation

**What it does:** scores every held-out passage's candidate pool with the fine-tuned Laya and with MiniLM (`cross-encoder/ms-marco-MiniLM-L-6-v2`, loaded the way the RAG loads it), reads the teacher's stored probabilities, and computes nDCG@10 for each with a paired bootstrap interval on the differences, the calibration of Laya's P(true), a collapse check and the scoring latency. It writes `metrics.json` next to the checkpoint.

**nDCG@10, and what "relevant" means here.** A held-out passage's pool is itself plus its hard negatives. The relevant candidate is the passage the question was written from (binary gain). That under-counts negatives that also answer the question, but it does so identically for all three scorers. The teacher's ordering is the ceiling to compare with, because a student cannot beat its teacher on the teacher's own labels; each scorer's nDCG is also reported with the teacher's probability as the gain, i.e. how closely it follows the teacher. **Rule of thumb for a spike:** if Laya's nDCG@10 is not above MiniLM's, with the paired interval excluding zero, stop. The held-out passages come from the same distribution as the training ones; the repo's bench on the human-reviewed eval questions is the real test (manual, section 5).

**Calibration is what the relevance-gate role needs.** A gate thresholds P(true), so P(true) has to mean something. `reliability_bins` cuts P(true) into 10 equal bins and reports, per bin, how many predictions fell in it, their mean, and the observed rate of positives; ECE is the bin-weighted gap, and the cell checks it against `laya.common.ece_score` (it raises if they disagree). It is computed against two labels: the source passage (a proxy for "relevant") and the teacher's own hard label. Upstream's published ECE is on the answer confidence `max(p, 1-p)`; that is reported too, labelled as such, but a gate needs the first two. The teacher's ECE against the same source label is printed for reference, since the student's calibration is bounded by it. Laya's P(true) is rounded to 4 decimals, so a threshold finer than that means nothing.

**Collapse check.** If the standard deviation of the student's P(true), or its mean within-query range, is under `COLLAPSE_RATIO` (0.2) of the teacher's, the student has collapsed (see cell 7). The cell writes `metrics.json` first, then raises. 0.2 is a tripwire chosen for this notebook, not an upstream number.

**Latency** is wall-clock per candidate pool on this GPU for Laya, MiniLM and the teacher. It is a hint, not the graduation criterion, which is p95 CPU rerank latency measured by the bench. The cell also compares batch-of-16 scores with one-at-a-time scores (third parties reported padded-batch NaNs on CUDA) and raises on a NaN.

In [ ]:
# Cell 8 - held-out evaluation. The ECE helper (laya.common.ece_score) and the idea of scoring the fine-tune by ECE next to
# accuracy come from NandhaKishorM/laya (Apache-2.0); the ranking metrics, reliability bins and collapse check are this task's.
import math
from FlagEmbedding import FlagReranker
from laya.common import ece_score
from sentence_transformers import CrossEncoder

MINILM_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"    # the reranker the RAG ships (retrieval.cross_encoder_model)
LAYA_BATCH = 16                                       # the inference adapter's default batch size
LAT_QUERIES = 100                                     # queries timed per scorer, after 3 warm-up calls
N_BINS = 10
COLLAPSE_RATIO = 0.2                                  # a tripwire, not a quality bar: see the markdown above
BOOTSTRAP = 10_000


# Pure helpers: unit-tested by tests/test_laya_notebook.py, which pulls them out of this notebook and runs them.
def ndcg_at_k(scores, gains, k=10):
    """nDCG@k of the order `scores` induce over `gains`. The sort is stable, so equal scores keep the candidate order."""
    scores, gains = np.asarray(scores, dtype=float), np.asarray(gains, dtype=float)
    top = np.argsort(-scores, kind="stable")[:k]
    discount = 1.0 / np.log2(np.arange(2, len(top) + 2))
    ideal = np.sort(gains)[::-1][:k]
    ideal_dcg = float((ideal * discount[:len(ideal)]).sum())
    return float((gains[top] * discount).sum()) / ideal_dcg if ideal_dcg > 0 else 0.0


def reliability_bins(p, y, bins=N_BINS):
    """Equal-width bins of P(true): how many predictions fall in each, their mean, and the observed rate of positives."""
    p, y = np.asarray(p, dtype=float), np.asarray(y, dtype=float)
    edges, rows = np.linspace(0.0, 1.0, bins + 1), []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        inside = (p >= lo if i == 0 else p > lo) & (p <= hi)       # the same edges as laya's ece_score
        if inside.any():
            rows.append({"lo": float(lo), "hi": float(hi), "n": int(inside.sum()),
                         "mean_p": float(p[inside].mean()), "frac_true": float(y[inside].mean())})
    return rows


def ece_from_bins(rows):
    """Expected calibration error: the bin-weighted gap between what was predicted and what was observed."""
    n = sum(r["n"] for r in rows)
    return sum(r["n"] / n * abs(r["mean_p"] - r["frac_true"]) for r in rows)


def paired_bootstrap_ci(diffs, n_boot=BOOTSTRAP, seed=0):
    """(mean, lo, hi): the mean paired difference and its 95% bootstrap interval over queries, fixed seed."""
    d = np.asarray(diffs, dtype=float)
    picks = np.random.default_rng(seed).integers(0, len(d), size=(n_boot, len(d)))
    lo, hi = np.quantile(d[picks].mean(axis=1), [0.025, 0.975])
    return float(d.mean()), float(lo), float(hi)


def collapse_check(student, teacher, min_ratio=COLLAPSE_RATIO):
    """student, teacher: one list of P(true) per query. A collapsed student gives (almost) the same answer for every
    input, so both its overall spread and its spread within a query fall far below the teacher's."""
    s_all, t_all = np.concatenate(student), np.concatenate(teacher)
    out = {"student_std": float(s_all.std()), "teacher_std": float(t_all.std()),
           "student_query_range": float(np.mean([max(x) - min(x) for x in student])),
           "teacher_query_range": float(np.mean([max(x) - min(x) for x in teacher]))}
    out["std_ratio"] = out["student_std"] / out["teacher_std"] if out["teacher_std"] > 0 else float("nan")
    out["range_ratio"] = out["student_query_range"] / out["teacher_query_range"] if out["teacher_query_range"] > 0 else float("nan")
    out["collapsed"] = bool(out["std_ratio"] < min_ratio or out["range_ratio"] < min_ratio)
    return out


def time_per_query(fn, groups, warmup=3):
    """Wall-clock milliseconds to score one query's candidate pool, p50 / p95 / mean."""
    for g in groups[:warmup]:
        fn(g["query"], g["texts"])
    ms = []
    for g in groups:
        t0 = time.perf_counter()
        fn(g["query"], g["texts"])
        ms.append((time.perf_counter() - t0) * 1000)
    return {"p50_ms": float(np.percentile(ms, 50)), "p95_ms": float(np.percentile(ms, 95)), "mean_ms": float(np.mean(ms)),
            "queries": len(ms), "candidates_per_query": len(groups[0]["texts"])}


# CKPT_DIR is absolute on purpose: laya.load reads a missing relative path as a Hugging Face repo id, not as an error
agent = laya.load(CKPT_DIR, device="cuda:0")
cross = CrossEncoder(MINILM_ID, max_length=512, device="cuda:0")
truncated = []


def laya_scores(query, texts):
    out = agent.predict_batch(texts, make_question(query), batch_size=LAYA_BATCH)
    truncated.extend(bool(r["usage"]["truncated"]) for r in out)
    p = [r["answers"]["rel"]["noul"] for r in out]
    if any(math.isnan(x) for x in p):
        raise ValueError("Laya returned NaN for a held-out pair (padded-batch NaNs have been reported on CUDA): set LAYA_BATCH = 1 and rerun")
    return p


def minilm_scores(query, texts):
    return cross.predict([(query, t) for t in texts]).tolist()


for g in heldout:
    g["laya"] = laya_scores(g["query"], g["texts"])
    g["minilm"] = minilm_scores(g["query"], g["texts"])
truncated_share = float(np.mean(truncated))

# ranking: relevance is "this is the passage the question was written from" (binary), and, separately, the teacher's own P
binary = {name: np.array([ndcg_at_k(g[key], g["label"]) for g in heldout]) for name, key in
          (("laya", "laya"), ("minilm", "minilm"), ("teacher", "teacher_p"))}
follows_teacher = {name: np.array([ndcg_at_k(g[name], g["teacher_p"]) for g in heldout]) for name in ("laya", "minilm")}
ndcg = {
    "n_queries": len(heldout),
    "binary_gain": {k: float(v.mean()) for k, v in binary.items()},
    "teacher_p_as_gain": {k: float(v.mean()) for k, v in follows_teacher.items()},
    "paired_diff_binary_gain": {
        "laya_minus_minilm": dict(zip(("mean", "lo", "hi"), paired_bootstrap_ci(binary["laya"] - binary["minilm"]))),
        "laya_minus_teacher": dict(zip(("mean", "lo", "hi"), paired_bootstrap_ci(binary["laya"] - binary["teacher"]))),
    },
}

# calibration of P(true): against the source-passage label and against the teacher's own hard label
p_all = np.concatenate([g["laya"] for g in heldout])
t_all = np.concatenate([g["teacher_p"] for g in heldout])
y_source = np.concatenate([g["label"] for g in heldout]).astype(float)
y_teacher = (t_all >= 0.5).astype(float)
bins_source, bins_teacher = reliability_bins(p_all, y_source), reliability_bins(p_all, y_teacher)
calibration = {
    "vs_source_passage": {"ece": ece_from_bins(bins_source), "bins": bins_source},
    "vs_teacher_label": {"ece": ece_from_bins(bins_teacher), "bins": bins_teacher},
    "teacher_vs_source_passage_ece": ece_from_bins(reliability_bins(t_all, y_source)),
    "ece_of_answer_confidence_upstream_definition": float(ece_score(np.maximum(p_all, 1 - p_all), ((p_all >= 0.5) == (y_teacher == 1)).astype(float))),
    "mean_abs_error_vs_teacher_p": float(np.abs(p_all - t_all).mean()),
}
assert abs(calibration["vs_source_passage"]["ece"] - ece_score(p_all, y_source, bins=N_BINS)) < 1e-9, \
    "our reliability bins disagree with laya's ece_score"
collapse = collapse_check([g["laya"] for g in heldout], [g["teacher_p"] for g in heldout])

# batch parity (the NaN/padding hazard third parties reported on CUDA) and scoring latency, all on this GPU
probe = heldout[0]
parity = max(abs(a - b) for a, b in zip(laya_scores(probe["query"], probe["texts"]),
                                        [laya_scores(probe["query"], [t])[0] for t in probe["texts"]]))
teacher = FlagReranker(TEACHER_ID, use_fp16=True, devices="cuda:0")
latency = {
    "laya": time_per_query(laya_scores, heldout[:LAT_QUERIES]),
    "minilm": time_per_query(minilm_scores, heldout[:LAT_QUERIES]),
    "teacher": time_per_query(lambda q, texts: teacher.compute_score([(q, t) for t in texts], normalize=True, batch_size=TEACHER_BATCH),
                              heldout[:LAT_QUERIES]),
    "device": torch.cuda.get_device_name(0),
    "note": "GPU scoring time per candidate pool. The graduation rule in the manual uses p95 CPU rerank latency: measure that with the bench.",
}
del teacher, agent, cross
gc.collect()
torch.cuda.empty_cache()

metrics = {"provenance": RUN, "ndcg_at_10": ndcg, "calibration": calibration, "collapse": collapse, "latency": latency,
           "laya_truncated_share": truncated_share, "batch_vs_single_max_abs_diff": parity}
with open(f"{CKPT_DIR}/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"nDCG@10 over {len(heldout)} held-out queries (binary gain = the passage the question came from):")
for name, v in ndcg["binary_gain"].items():
    print(f"  {name:8s}{v:.3f}")
for name, d in ndcg["paired_diff_binary_gain"].items():
    print(f"  {name}: {d['mean']:+.3f}  95% CI [{d['lo']:+.3f}, {d['hi']:+.3f}]")
print(f"ECE of P(true): {calibration['vs_source_passage']['ece']:.3f} vs the source passage, "
      f"{calibration['vs_teacher_label']['ece']:.3f} vs the teacher's label (teacher itself: {calibration['teacher_vs_source_passage_ece']:.3f})")
for r in bins_source:
    print(f"  P(true) {r['lo']:.1f}-{r['hi']:.1f}: n={r['n']:5d}  mean {r['mean_p']:.3f}  observed {r['frac_true']:.3f}")
print(f"latency p50/p95 ms per pool of {latency['laya']['candidates_per_query']}: " +
      ", ".join(f"{k} {latency[k]['p50_ms']:.0f}/{latency[k]['p95_ms']:.0f}" for k in ("laya", "minilm", "teacher")) + f"  ({latency['device']})")
print(f"truncated passages {truncated_share:.1%}; batch vs single max |diff| {parity:.4f}; collapse check {collapse}")
if collapse["collapsed"]:
    raise RuntimeError("the student has collapsed (it answers almost the same for every input). metrics.json holds the evidence; "
                       "see the collapse paragraph (upstream issue 963) in cell 7's markdown before spending another run.")

## 9. Save and zip

**What it does:** re-checks the checkpoint (the four entries `laya.load` reads, and that the saved config says 512 / 192), copies it with `metrics.json` into `/kaggle/working/noetrix-laya`, and zips that to `/kaggle/working/noetrix-laya.zip`.

**Why:** the zip's root *is* the checkpoint, so unzipping it into `models/laya-noetrix` gives exactly the directory the RAG's `retrieval.laya.model_dir` expects: `model.safetensors`, `encoder/`, `tokenizer/`, `rl_agent_config.json`, plus `metrics.json`. Nothing else is copied. The rows, the tokenised items and the rolling checkpoint stay in `SCRATCH`, because they hold vault text or duplicate the weights.

**There is deliberately no upload cell.** Upstream's notebook ends by publishing the model to a *public* Hugging Face repository with a write token. A model trained on vault passages must not leave your machine that way. Download the zip by hand (the notebook's Output tab, or `kaggle kernels output`), then follow section 6 of the manual to delete the dataset and this notebook's saved versions.

In [ ]:
# Cell 9 - save the checkpoint and metrics.json to /kaggle/working/noetrix-laya and zip it.
problem = checkpoint_problem(CKPT_DIR)
if problem:
    raise RuntimeError(f"not shipping: {problem}")
with open(f"{CKPT_DIR}/rl_agent_config.json", encoding="utf-8") as f:
    saved = json.load(f)
for key, want in (("max_len", MAX_LEN), ("head_max_len", HEAD_MAX_LEN)):
    if saved.get(key) != want:
        raise RuntimeError(f"the saved config says {key} = {saved.get(key)} but the model was trained at {want}")
if not os.path.isfile(f"{CKPT_DIR}/metrics.json"):
    raise RuntimeError("metrics.json is missing: run cell 8 first")

if os.path.isdir(OUT_DIR):
    shutil.rmtree(OUT_DIR)
shutil.copytree(CKPT_DIR, OUT_DIR)            # exactly the files laya.load reads, plus metrics.json
zip_path = shutil.make_archive(f"{WORK_DIR}/noetrix-laya", "zip", root_dir=OUT_DIR)    # archive root = the checkpoint files

for root, _, files in sorted(os.walk(OUT_DIR)):
    for name in sorted(files):
        path = os.path.join(root, name)
        print(f"{os.path.getsize(path) / 2**20:9.1f} MiB  {os.path.relpath(path, OUT_DIR)}")
print(f"\n{zip_path}: {os.path.getsize(zip_path) / 2**30:.2f} GiB")
print("Next: download that zip, unzip it into models/laya-noetrix, then DELETE the Kaggle dataset and this notebook's saved "
      "versions (docs/laya-finetune.md, section 6).")